# Section 4.6

# Governance Misfit

## Objective

To evaluate how governance decisions can become misaligned with empirical model outputs, resulting in policy misfit and resource allocation inefficiencies.

This experiment operationalises the theoretical concept of Governance Misfit developed in Section 4.6 of the paper.

## Research Question

How do governance policies and operational resource constraints influence the effectiveness of algorithmic decision-making?

## Theory Link

Section 3 argues that governance systems frequently rely on simplified representations of complex social phenomena. When institutional policies or operational constraints fail to align with empirical evidence, governance decisions may systematically diverge from optimal outcomes.

This notebook evaluates that claim through simulations of policy threshold misalignment and constrained review capacity.

## Dataset Description

Dataset: Heart Disease Prediction Dataset

Target Variable:

- HeartDisease

Selected Features:

- Age
- Sex
- ChestPainType
- RestingBP
- Cholesterol
- FastingBS
- RestingECG
- MaxHR
- ExerciseAngina
- Oldpeak
- ST_Slope

The dataset provides a proxy environment for evaluating governance decision-making under policy and resource constraints.

In [1]:
#Imports
# Imports

import sys
from pathlib import Path

# Make project root importable
sys.path.append(str(Path.cwd().parent))

import pandas as pd

from src.config import (
    HEART_DATA_PATH,
    FIGURES_DIR,
    TABLES_DIR,
)

from src.utils import (
    load_data,
    save_dataframe,
)

from src.preprocessing import (
    encode_categorical_variables,
    split_features_target,
)

from src.modelling import (
    split_train_test,
    train_logistic_regression,
    predict_model,
)

from src.governance import (
    simulate_policy_misfit,
    simulate_resource_constraints,
    create_governance_summary,
    plot_governance_tradeoff,
)

In [2]:
#Configuration
# Configuration

RANDOM_STATE = 42

In [3]:
#Data Loading
# Load dataset

df = load_data(HEART_DATA_PATH)

df.shape

(918, 12)

In [4]:
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [5]:
#Preprocessing
encoded_df = encode_categorical_variables(df)

encoded_df.head()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease,Sex_M,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_Normal,RestingECG_ST,ExerciseAngina_Y,ST_Slope_Flat,ST_Slope_Up
0,40,140,289,0,172,0.0,0,True,True,False,False,True,False,False,False,True
1,49,160,180,0,156,1.0,1,False,False,True,False,True,False,False,True,False
2,37,130,283,0,98,0.0,0,True,True,False,False,False,True,False,False,True
3,48,138,214,0,108,1.5,1,False,False,False,False,True,False,True,True,False
4,54,150,195,0,122,0.0,0,True,False,True,False,True,False,False,False,True


In [6]:
X, y = split_features_target(
    encoded_df,
    "HeartDisease",
)

In [7]:
X_train, X_test, y_train, y_test = split_train_test(
    X,
    y,
)

In [8]:
model = train_logistic_regression(
    X_train,
    y_train,
)

In [9]:
y_pred, y_prob = predict_model(
    model,
    X_test,
)

# Scenario A — Policy Misfit

Governance systems often apply fixed policy thresholds that do not perfectly align with empirical model outputs.

This experiment evaluates how increasing policy strictness affects the proportion of cases approved under the governance framework.

In [10]:
policy_table = simulate_policy_misfit(
    y_prob,
)

In [11]:
policy_table

,Probability,BaselineDecision,PolicyMisfitDecision
0,0.978639,1,1
1,0.148080,0,0
2,0.960518,1,1
3,0.582410,1,0
4,0.172495,0,0
...,...,...,...
179,0.190033,0,0
180,0.975535,1,1
181,0.880174,1,1
182,0.942605,1,1


# Scenario B — Resource Constraints

Many governance systems cannot review every case because of staffing, budget, or operational limitations.

This experiment simulates how varying review capacity influences the proportion of cases that can be processed.

In [13]:
governance_df = simulate_policy_misfit(
    y_prob,
)

governance_df.head()

,Probability,BaselineDecision,PolicyMisfitDecision
0,0.978639,1,1
1,0.148080,0,0
2,0.960518,1,1
3,0.582410,1,0
4,0.172495,0,0


In [14]:
governance_df = simulate_resource_constraints(
    governance_df,
)

governance_df.head()

,Probability,BaselineDecision,PolicyMisfitDecision,Reviewed
0,0.978639,1,1,True
1,0.148080,0,0,False
2,0.960518,1,1,True
3,0.582410,1,0,True
4,0.172495,0,0,True


In [15]:
governance_summary = create_governance_summary(
    governance_df,
)

governance_summary

,Metric,Value
0,Total Cases,184
1,Baseline Positive Decisions,109
2,Policy Misfit Decisions,100
3,Reviewed Cases,138
4,Unreviewed Cases,46


In [16]:
save_dataframe(
    governance_summary,
    TABLES_DIR / "table_8_governance_summary.csv",
)

In [17]:
plot_governance_tradeoff(
    governance_summary,
    FIGURES_DIR / "figure_8_governance_tradeoff.png",
)

# Results Interpretation

The governance simulations demonstrate that institutional constraints can alter algorithmic decision outcomes even when model performance remains constant.

The policy misfit scenario illustrates how governance thresholds influence classification decisions, while the resource constraint simulation shows that limited review capacity leaves a substantial proportion of cases without governance oversight.

These findings reinforce the argument that governance effectiveness depends not only on predictive performance but also on organisational policies and operational resources.

# Outputs Generated

This notebook produced:

- Table 8 — Governance Misfit Summary
- Figure 8 — Governance Trade-offs

All outputs were saved to the project's outputs directory for inclusion in the manuscript and interactive dashboard.

# Conclusion

This notebook operationalised Governance Misfit by simulating policy threshold shifts and resource limitations.

The experiments demonstrate that governance outcomes depend on both institutional policy and operational capacity. Even highly accurate predictive systems can experience governance failures when review resources are constrained or decision policies become misaligned.

These findings provide empirical support for governance-aware AI deployment and complete the empirical validation pipeline for the simulated subjecthood framework.